# 🎯 Workspace Architecture & System Prompt

> **System Prompt / Meta-Prompt Specification**
> 
> You are an AI assistant managing the `/home/dev/SE` workspace environment for user **devkumar70401**.
>
> ### Workspace Guiding Principles & Architecture:
> 1. **Root Directory Role (`SE`)**:
>    - The `/home/dev/SE` directory is a workspace container folder, **NOT** a Git repository itself.
>    - Do NOT initialize a `.git` repository at the root `/home/dev/SE` level.
>    - Workspace multi-root structure is defined in [`SE.code-workspace`](file:///home/dev/SE/SE.code-workspace).
> 
> 2. **Modular Multi-Repository Structure**:
>    - Every subfolder within `SE/` represents an independent GitHub repository belonging to account **devkumar70401** (`git@github.com:devkumar70401/<repo>.git`).
>    - **`Devendra/`**: Personal files, private configs, and personal workspace data.
>    - **`Library/`**: Reference materials including PDFs, documents, books, e-books, research papers, and asset resources.
>    - **`Notes/`**: Personal notes, handwritten scans, markdown study logs (`1-CS/`, `2-ML/`, `3-Robotics/`).
>    - **`P1_Wine_Inventory_Management/`**: Project 1 (Wine Inventory Management).
>    - **`P2_Warpsync/`**: Project 2 (WarpSync: Local peer-to-peer file sharing application like ShareIt / LocalSend).
>    - **`P<N>_<Project_Name>/`**: Future projects following this standard naming scheme.
> 
> 3. **Git & Security Protocol (SSH / SSL Authentication)**:
>    - **ALWAYS** use SSH URLs (`git@github.com:devkumar70401/<repo>.git`) instead of HTTPS (`https://github.com/...`).
>    - User email: `devkumar70401@gmail.com`.
>    - Authenticate using SSH keypairs (`ed25519` or `rsa-4096`) registered with GitHub to enable seamless, passwordless pushes and pulls.
>    - Maintain strict `.gitignore` configurations per repository.
> 
> 4. **Project Lifecycle & Automation**:
>    - When creating a new project, follow the naming scheme `P<N>_<Project_Name>`.
>    - Automatically run `git init -b main` within the new directory and link it to its corresponding GitHub repository via SSH remote `git@github.com:devkumar70401/<repo_name>.git`.


## 📁 Workspace Overview & Verification

This notebook guides you through setting up and managing your workspace structure under `/home/dev/SE`.
The code block below inspects each folder in your workspace to verify that it is an independent Git repository.


In [ ]:
import os
import subprocess

workspace_dir = '/home/dev/SE'
subfolders = [f for f in os.listdir(workspace_dir) if os.path.isdir(os.path.join(workspace_dir, f)) and not f.startswith('.')]

print(f"Found {len(subfolders)} subfolders in workspace '{workspace_dir}':\n")
for folder in sorted(subfolders):
    folder_path = os.path.join(workspace_dir, folder)
    is_git = os.path.isdir(os.path.join(folder_path, '.git'))
    remotes = ''
    if is_git:
        try:
            remotes = subprocess.check_output(['git', 'remote', '-v'], cwd=folder_path, text=True).strip()
        except Exception:
            remotes = 'No remotes configured'
    print(f"- {folder:30s} | Git Repo: {'YES' if is_git else 'NO '}")
    if remotes:
        print(f"  Remote: {remotes}")


## 🔑 Step 1: Set Up SSH Key Authentication (SSH/SSL Protocol)

Using SSH (`git@github.com:...`) instead of HTTPS (`https://github.com/...`) is secure and allows seamless `git push`/`git pull` without entering personal access tokens repeatedly.

### Instructions:
1. Run the code cell below to generate an Ed25519 SSH key for **devkumar70401@gmail.com** (if you don't already have one).
2. Copy the public key output.
3. Go to GitHub -> **Settings** -> **SSH and GPG keys** -> **New SSH key** and paste your public key.
4. Test your connection with `ssh -T git@github.com`.


In [ ]:
import os
import subprocess

ssh_dir = os.path.expanduser('~/.ssh')
key_path = os.path.join(ssh_dir, 'id_ed25519')
pub_key_path = key_path + '.pub'
email = 'devkumar70401@gmail.com'

# Generate SSH key if it doesn't exist
if not os.path.exists(key_path):
    os.makedirs(ssh_dir, mode=0o700, exist_ok=True)
    subprocess.run(['ssh-keygen', '-t', 'ed25519', '-C', email, '-f', key_path, '-N', ''], check=True)
    print('✅ SSH Key generated successfully for ' + email)
else:
    print('ℹ️ SSH Key already exists.')

if os.path.exists(pub_key_path):
    with open(pub_key_path, 'r') as f:
        pub_key = f.read().strip()
    print('\n📋 COPY THIS PUBLIC KEY TO GITHUB (https://github.com/settings/keys):\n')
    print(pub_key)


## 🔗 Step 2: Configure SSH Remotes for Repositories

Set the remote origin for each repository using SSH format: `git@github.com:devkumar70401/<repo_name>.git`.


In [ ]:
import os
import subprocess

GITHUB_USERNAME = 'devkumar70401'
workspace_dir = '/home/dev/SE'

# Map directories to their GitHub repository names
repo_mappings = {
    'Devendra': 'Devendra',
    'Library': 'Library',
    'Notes': 'Notes',
    'P1_Wine_Inventory_Management': 'P1_Wine_Inventory_Management',
    'P2_Warpsync': 'P2_Warpsync'
}

for folder, repo_name in repo_mappings.items():
    folder_path = os.path.join(workspace_dir, folder)
    if os.path.exists(folder_path):
        ssh_url = f'git@github.com:{GITHUB_USERNAME}/{repo_name}.git'
        # Ensure git repo is initialized
        subprocess.run(['git', 'init', '-b', 'main'], cwd=folder_path, stdout=subprocess.DEVNULL)
        
        # Check if origin remote exists
        remotes = subprocess.check_output(['git', 'remote'], cwd=folder_path, text=True).split()
        if 'origin' in remotes:
            subprocess.run(['git', 'remote', 'set-url', 'origin', ssh_url], cwd=folder_path)
            print(f"Updated origin for '{folder}' -> {ssh_url}")
        else:
            subprocess.run(['git', 'remote', 'add', 'origin', ssh_url], cwd=folder_path)
            print(f"Added origin for '{folder}' -> {ssh_url}")


## 🚀 Step 3: Automate Creation of New Projects (`P3_...`, `P4_...`)

Use the Python function below to easily instantiate new project repositories in your workspace following your naming conventions.


In [ ]:
import os
import subprocess

def create_new_project(project_num: int, project_name: str, github_username: str = 'devkumar70401'):
    """
    Creates a new project directory P<num>_<project_name>, initializes git with 'main' branch,
    creates a README.md and .gitignore, and sets up GitHub SSH remote.
    """
    formatted_name = f"P{project_num}_{project_name.strip().replace(' ', '_')}"
    project_path = os.path.join('/home/dev/SE', formatted_name)
    
    os.makedirs(project_path, exist_ok=True)
    subprocess.run(['git', 'init', '-b', 'main'], cwd=project_path, check=True)
    
    # Create default README.md
    readme_path = os.path.join(project_path, 'README.md')
    if not os.path.exists(readme_path):
        with open(readme_path, 'w') as f:
            f.write(f'# {formatted_name}\n\nProject description for {project_name}.\n')
            
    # Create default .gitignore
    gitignore_path = os.path.join(project_path, '.gitignore')
    if not os.path.exists(gitignore_path):
        with open(gitignore_path, 'w') as f:
            f.write('__pycache__/\n*.pyc\n.env\n.vscode/\n.idea/\nbuild/\ndist/\nnode_modules/\n')
            
    ssh_url = f'git@github.com:{github_username}/{formatted_name}.git'
    subprocess.run(['git', 'remote', 'add', 'origin', ssh_url], cwd=project_path)
    print(f'✅ Project created: {formatted_name} with SSH remote {ssh_url}')

# Example usage:
# create_new_project(3, 'Smart_Recommendation_Engine')
